# Table 2: Proposed-model configuration ablation

Chạy sáu cấu hình của Proposed trên cùng dữ liệu hai camera:
- **Local Belief only**
- **Without Conflict Detection**
- **Without H_th**
- **Without L_temp**
- **Without Kinematic Constraints**
- **Full model**

Cả sáu cấu hình dùng chung `ALPHA = 0.01` và `BETA = 0.85` khai báo tại Cell 1.

Chọn **Runtime → Run all** để chạy toàn bộ pipeline. Notebook sẽ bỏ qua cấu hình đã hoàn tất và tiếp tục đúng worksheet đang dở.

> Google Colab vẫn yêu cầu xác nhận quyền truy cập Drive/Sheets; đây là bước bảo mật không thể tự động bỏ qua.


In [ ]:
#@title 1. Cấu hình chạy
REPO_URL = "https://github.com/doantrunghieu08/optimization_model_monocular_3.git"  #@param {type:"string"}
REPO_BRANCH = "version_raycasting"  #@param {type:"string"}
DATA_ROOT = "/content/drive/MyDrive"  #@param {type:"string"}
SUBJECTS = "S8"  #@param {type:"string"}
SEQUENCES = "Seq1"  #@param {type:"string"}
SEGMENTS = "*"  #@param {type:"string"}
EXCLUDED_CAMERAS = "1"  #@param {type:"string"}
FUSION_METHODS = "proposed"  # Table 2 chỉ ablate Proposed
ENABLE_LEARNABLE = False  #@param {type:"boolean"}
ENABLE_LEARNABLE_EXTRA = False  #@param {type:"boolean"}
SPREADSHEET_NAME = ""  #@param {type:"string"}
ALPHA = 0.01  #@param {type:"number"}
BETA = 0.85  #@param {type:"number"}
NOTEBOOK_NAME = "table2_ablation_optical_global_kinematic_huber_alpha1E_2_beta85E_2.ipynb"
SMPL_NEUTRAL_FILE_ID = "1xblXsbK1rTSn5cG934cDhRFB0Apn64Ll"  #@param {type:"string"}
SMPL_PART_SEGMENTATION_FILE_ID = "19w6RSoqdCJwUMu8wd1tYiF7uqLMO19BD"  #@param {type:"string"}

In [ ]:
#@title 2. Chuẩn bị repository và môi trường
import importlib
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path



def run(command, cwd=None):
    print("$", " ".join(map(str, command)))
    subprocess.run(command, cwd=cwd, check=True)


repo_dir = Path("/content/optimization_model_monocular_3")
if (repo_dir / ".git").exists():
    print(f"Updating branch: {REPO_BRANCH}")

    # Fetch branch và tạo/cập nhật chính xác remote-tracking ref
    run([
        "git",
        "fetch",
        "origin",
        f"{REPO_BRANCH}:refs/remotes/origin/{REPO_BRANCH}"
    ], repo_dir)

    # Tạo/reset local branch theo remote
    run([
        "git",
        "checkout",
        "-f",
        "-B",
        REPO_BRANCH,
        f"origin/{REPO_BRANCH}"
    ], repo_dir)
elif repo_dir.exists():
    backup_dir = repo_dir.with_name(f"{repo_dir.name}_backup_{os.getpid()}")
    repo_dir.rename(backup_dir)
    print(f"Moved the existing non-Git directory to {backup_dir}")
    run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(repo_dir)])
else:
    run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(repo_dir)])

# Install everything before importing project/Colab dependencies.
requirements_path = repo_dir / "requirements.txt"
if sys.version_info >= (3, 13):
    requirements_lines = requirements_path.read_text(encoding="utf-8").splitlines()
    requirements_lines = [
        line for line in requirements_lines
        if not re.match(r"^\s*(numpy|scipy)(?:[<>=!~;]|$)", line, re.IGNORECASE)
    ]
    requirements_lines.extend(["numpy>=2.1,<2.3", "scipy>=1.14.1,<1.15"])
    requirements_path = Path("/tmp/requirements-colab.txt")
    requirements_path.write_text("\n".join(requirements_lines) + "\n", encoding="utf-8")

run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements_path)])
run([
    sys.executable, "-m", "pip", "install", "-q",
    "gspread", "google-api-python-client", "pandas", "gdown", "PyYAML",
])

scientific_check = [
    sys.executable,
    "-c",
    "import numpy, scipy; from scipy.spatial.distance import cdist; "
    "print('NumPy', numpy.__version__, '| SciPy', scipy.__version__)",
]
check_result = subprocess.run(scientific_check, text=True, capture_output=True)
if check_result.returncode:
    print("Repairing the incompatible NumPy/SciPy installation...")
    scientific_specs = (
        ["numpy>=2.1,<2.3", "scipy>=1.14.1,<1.15"]
        if sys.version_info >= (3, 13)
        else ["numpy<2", "scipy<2"]
    )
    run([
        sys.executable, "-m", "pip", "install", "-q", "--upgrade",
        "--force-reinstall", "--no-cache-dir", *scientific_specs,
    ])
    run(scientific_check)
else:
    print(check_result.stdout.strip())

if shutil.which("ffmpeg") is None:
    run(["apt-get", "update", "-qq"])
    run(["apt-get", "install", "-y", "-qq", "ffmpeg"])
importlib.invalidate_caches()


In [ ]:
#@title 3. Kết nối Google Drive và tải model
from google.colab import auth, drive
from google.auth import default
from googleapiclient.discovery import build
import gdown
import yaml
from ruamel.yaml import YAML

drive.mount("/content/drive", force_remount=False)
auth.authenticate_user()
credentials, _ = default()

try:
    user = build("drive", "v3", credentials=credentials).about().get(fields="user").execute()["user"]
    os.environ["RUNNER_NAME"] = user.get("displayName", "Colab_User")
    os.environ["RUNNER_EMAIL"] = user.get("emailAddress", "")
except Exception as exc:
    print(f"Could not read Drive profile ({exc}); using Colab_User.")
    os.environ["RUNNER_NAME"] = "Colab_User"

model_path = repo_dir / "models" / "SMPL_NEUTRAL.pkl"
model_path.parent.mkdir(parents=True, exist_ok=True)
if not model_path.exists() or model_path.stat().st_size < 1_000_000:
    print("Downloading SMPL_NEUTRAL.pkl...")
    downloaded = gdown.download(id=SMPL_NEUTRAL_FILE_ID, output=str(model_path), quiet=False)
    if not downloaded or not model_path.exists():
        raise RuntimeError("Could not download SMPL_NEUTRAL.pkl. Check the Drive file ID/access permission.")

segmentation_path = repo_dir / "models" / "smpl_partSegmentation_mapping.pkl"
if not segmentation_path.exists() or segmentation_path.stat().st_size == 0:
    print("Downloading smpl_partSegmentation_mapping.pkl...")
    downloaded = gdown.download(id=SMPL_PART_SEGMENTATION_FILE_ID, output=str(segmentation_path), quiet=False)
    if not downloaded or not segmentation_path.exists() or segmentation_path.stat().st_size == 0:
        raise RuntimeError("Could not download smpl_partSegmentation_mapping.pkl. Check the Drive file ID/access permission.")


In [ ]:
#@title 4. Áp dụng cấu hình pipeline
# Table 2 ghi trực tiếp cấu hình vào pipeline.yml. Tên notebook chỉ dùng để đặt tên báo cáo.
# Vì vậy đổi tên file không thể âm thầm đổi ALPHA/BETA của sáu phép ablation.

import os
os.environ["NOTEBOOK_NAME"]          = NOTEBOOK_NAME
os.environ["ENABLE_LEARNABLE"]       = str(ENABLE_LEARNABLE).lower()
os.environ["ENABLE_LEARNABLE_EXTRA"] = str(ENABLE_LEARNABLE_EXTRA).lower()

# Patch pipeline.yml: bật optimizer cho ablation và cập nhật learnable flags.
from pathlib import Path
from ruamel.yaml import YAML

pipeline_path = repo_dir / "configs" / "pipeline.yml"
roundtrip_yaml = YAML()
with pipeline_path.open("r", encoding="utf-8") as stream:
    pipeline_config = roundtrip_yaml.load(stream)
pipeline_config["learnable"]["enabled"] = bool(ENABLE_LEARNABLE)
pipeline_config["learnable_extra"]["enabled"] = bool(ENABLE_LEARNABLE_EXTRA)
pipeline_config["fusion"]["belief"].update({
    "alpha": float(ALPHA),
    "beta": float(BETA),
    "local_method": "optical_aware_belief",
    "global": True,
})
pipeline_config["fusion"]["optimization"].update({
    "enabled": True,
    "use_kinematic_constraints": True,
    "loss_type": "huber",
})
if ALPHA <= 0 or not 0 <= BETA <= 1:
    raise ValueError("ALPHA must be positive and BETA must be between 0 and 1.")

with pipeline_path.open("w", encoding="utf-8") as stream:
    roundtrip_yaml.dump(pipeline_config, stream)

keypoint_map_path = repo_dir / "configs" / "keypoints3D_map.yml"
with keypoint_map_path.open("r", encoding="utf-8") as stream:
    keypoint_map = roundtrip_yaml.load(stream)
all_joint_names = [item["name"] for item in keypoint_map["keypoints"]]
if len(keypoint_map.get("priority2", [])) != len(all_joint_names):
    keypoint_map["priority2"] = all_joint_names
    with keypoint_map_path.open("w", encoding="utf-8") as stream:
        roundtrip_yaml.dump(keypoint_map, stream)

In [ ]:
#@title 5. Dò dữ liệu và tạo brute_force.yml
def accepts(filter_text, value):
    requested = {item.strip() for item in filter_text.split(",") if item.strip()}
    return not requested or "*" in requested or value in requested


def find_video(image_sequence, segments_dir, pkl_path, camera_id, segment_id):
    candidates = [
        pkl_path.parent / "output.mp4",
        pkl_path.parent / f"video_{camera_id}_seg_{segment_id}.mp4",
        segments_dir / f"video_{camera_id}_seg_{segment_id}.mp4",
        image_sequence / f"video_{camera_id}.avi",
        image_sequence / f"video_{camera_id}.mp4",
    ]
    return next((path for path in candidates if path.exists()), None)


def find_image_sequences(root):
    if root.name == "imageSequence" and root.is_dir():
        return [root]
    subjects = [item.strip() for item in SUBJECTS.split(",") if item.strip() and item.strip() != "*"]
    sequences = [item.strip() for item in SEQUENCES.split(",") if item.strip() and item.strip() != "*"]
    if subjects and sequences:
        suffixes = [f"{subject}/{sequence}/imageSequence" for subject in subjects for sequence in sequences]
    else:
        suffixes = ["imageSequence"]
    patterns = [f"{'*/' * depth}{suffix}" for depth in range(4) for suffix in suffixes]
    return sorted({path for pattern in patterns for path in root.glob(pattern) if path.is_dir()})


data_root = Path(DATA_ROOT).expanduser()
if data_root.exists():
    image_sequences = find_image_sequences(data_root)
else:
    search_roots = [path for path in (Path("/content/drive/MyDrive"), Path("/content/drive/Shareddrives")) if path.is_dir()]
    image_sequences = sorted({path for root in search_roots for path in find_image_sequences(root)})
    if not image_sequences:
        raise FileNotFoundError(
            f"DATA_ROOT does not exist: {data_root}. No imageSequence directory was found in the mounted Drive. "
            "Set DATA_ROOT in cell 1 to the dataset folder or its imageSequence directory."
        )
    print(f"DATA_ROOT not found: {data_root}")
    print(f"Using {len(image_sequences)} imageSequence directorie(s) discovered in the mounted Drive.")

if not image_sequences:
    raise FileNotFoundError(
        f"No imageSequence directory was found under DATA_ROOT: {data_root}. "
        "Set DATA_ROOT in cell 1 to the dataset folder or its imageSequence directory."
    )
excluded_cameras = {item.strip() for item in EXCLUDED_CAMERAS.split(",") if item.strip()}
discovered_segments = []

for image_sequence in image_sequences:
    sequence = image_sequence.parent.name
    subject = image_sequence.parent.parent.name
    if not accepts(SUBJECTS, subject) or not accepts(SEQUENCES, sequence):
        continue

    segments_dir = next((image_sequence / name for name in ("Segments", "segments") if (image_sequence / name).is_dir()), None)
    gt_dir = next((path for path in (image_sequence / "GT", image_sequence.parent / "GT") if path.is_dir()), None)
    if segments_dir is None or gt_dir is None:
        print(f"Skipping {subject}/{sequence}: missing Segments or GT directory.")
        continue

    grouped = {}
    for pkl_path in sorted(segments_dir.rglob("*.pkl")):
        match = re.search(r"video_(\d+)_seg_(\d+)$", pkl_path.stem)
        if not match:
            continue
        camera_id, segment_id = match.groups()
        segment_name = f"seg_{segment_id}"
        if camera_id in excluded_cameras or not accepts(SEGMENTS, segment_name):
            continue
        if not (gt_dir / f"video_{camera_id}_{segment_name}.json").exists():
            continue
        video_path = find_video(image_sequence, segments_dir, pkl_path, camera_id, segment_id)
        if video_path is None:
            continue
        grouped.setdefault(segment_name, {})[camera_id] = {
            "id": f"video_{camera_id}_{segment_name}",
            "pkl": str(pkl_path.resolve()),
            "video": str(video_path.resolve()),
        }

    for segment_name, cameras_by_id in sorted(grouped.items()):
        cameras = [cameras_by_id[key] for key in sorted(cameras_by_id, key=int)]
        if len(cameras) >= 2:
            discovered_segments.append({
                "name": f"{subject}_{sequence}_{segment_name}",
                "ground_truth_dir": str(gt_dir.resolve()),
                "cameras": cameras,
            })

if not discovered_segments:
    raise RuntimeError(
        "No runnable segment with at least two cameras was found. "
        "Expected PKL names like video_0_seg_1.pkl and matching GT JSON files."
    )

fusion_methods = [item.strip() for item in FUSION_METHODS.split(",") if item.strip()]
allowed_methods = {"proposed", "aligned_averaging", "higher_belief_selection"}
if not fusion_methods or any(method not in allowed_methods for method in fusion_methods):
    raise ValueError(f"FUSION_METHODS must contain only: {', '.join(sorted(allowed_methods))}")

brute_force_path = repo_dir / "configs" / "brute_force.yml"
with brute_force_path.open("w", encoding="utf-8") as stream:
    yaml.safe_dump({"fusion_methods": fusion_methods, "segments": discovered_segments}, stream, sort_keys=False, allow_unicode=True)

camera_count = sum(len(segment["cameras"]) for segment in discovered_segments)
pair_count = sum(len(segment["cameras"]) * (len(segment["cameras"]) - 1) for segment in discovered_segments) * len(fusion_methods)
print(f"Discovered {len(discovered_segments)} segments, {camera_count} camera inputs, {pair_count} method/pair runs.")
print(f"Fusion methods: {fusion_methods}; learnable={ENABLE_LEARNABLE}, learnable_extra={ENABLE_LEARNABLE_EXTRA}")


In [ ]:
#@title 6. Chạy sáu cấu hình và tạo bảng tổng hợp
import copy
import statistics
from datetime import datetime

os.chdir(repo_dir)
sys.path.insert(0, str(repo_dir))
import brute_force_runner

run_tag = datetime.now().strftime("%y%m%d_%H%M%S")
report_name = SPREADSHEET_NAME.strip() or f"table2_ablation_alpha{ALPHA:g}_beta{BETA:g}"
base_pipeline_config = copy.deepcopy(pipeline_config)
ablations = [
    {
        "label": "Local Belief only",
        "slug": "01_local_belief_only",
        "overrides": {
            "fusion.belief.global": False,
            "fusion.occlusion.enabled": False,
            "fusion.correction.enabled": False,
            "fusion.correction.orientation_enabled": False,
            "fusion.optimization.regularization": False,
            "fusion.optimization.regularization_lambda": 0.0,
            "fusion.optimization.temporal_lambda": 0.0,
            "fusion.optimization.accel_lambda": 0.0,
        },
    },
    {
        "label": "Without Conflict Detection",
        "slug": "02_without_conflict_detection",
        "overrides": {"fusion.occlusion.enabled": False},
    },
    {
        "label": "Without H_th",
        "slug": "03_without_H_th",
        "overrides": {
            "fusion.optimization.regularization": False,
            "fusion.optimization.regularization_lambda": 0.0,
        },
    },
    {
        "label": "Without L_temp",
        "slug": "04_without_L_temp",
        "overrides": {
            "fusion.optimization.temporal_lambda": 0.0,
            "fusion.optimization.accel_lambda": 0.0,
        },
    },
    {
        "label": "Without Kinematic Constraints",
        "slug": "05_without_kinematic_constraints",
        "overrides": {"fusion.optimization.use_kinematic_constraints": False},
    },
    {
        "label": "Full model",
        "slug": "06_full_model",
        "overrides": {},
    },
]
assert len(ablations) == 6 and len({item['slug'] for item in ablations}) == 6

def apply_overrides(config, overrides):
    for dotted_key, value in overrides.items():
        target = config
        keys = dotted_key.split(".")
        for key in keys[:-1]:
            target = target[key]
        target[keys[-1]] = value


def write_pipeline(config):
    with pipeline_path.open("w", encoding="utf-8") as stream:
        roundtrip_yaml.dump(config, stream)


def mean_column(values, column):
    header = values[0]
    index = header.index(column)
    numbers = []
    for row in values[1:]:
        if not row or row[0] == "End" or index >= len(row):
            continue
        try:
            numbers.append(float(row[index].strip().replace(",", ".")))
        except (TypeError, ValueError):
            pass
    return round(statistics.fmean(numbers), 2) if numbers else "N/A"


def has_end(values):
    return any(row and row[0] == "End" for row in values[1:])


def report_matches(values, config):
    if len(values) < 2:
        return False
    indices = brute_force_runner._get_header_indices(values[0])
    required = ("Segment", "Cam Master", "Cam Slave", "All MPJPE")
    if any(indices[column] == -1 for column in required):
        return False
    results = []
    for row in values[1:]:
        try:
            _, result = brute_force_runner._parse_history_row(row, indices)
        except (IndexError, TypeError, ValueError):
            continue
        if result and result.get("mpjpe", float("inf")) != float("inf"):
            results.append(result)
    return bool(results) and all(brute_force_runner._matches_active_config(result, config) for result in results)


def find_report_worksheet(slug, config):
    try:
        spreadsheet = brute_force_runner.get_gspread_client().open(report_name)
    except brute_force_runner.gspread.exceptions.SpreadsheetNotFound:
        return None
    candidates = []
    for worksheet in reversed(spreadsheet.worksheets()):
        values = worksheet.get_all_values()
        named = worksheet.title == slug or worksheet.title.startswith(f"{slug}_")
        if not values:
            if named:
                candidates.append((False, 0, worksheet))
            continue
        if report_matches(values, config):
            row_count = sum(row and row[0] != "End" for row in values[1:])
            candidates.append((has_end(values), row_count, worksheet))
    completed = [candidate for candidate in candidates if candidate[0]]
    pool = completed or candidates
    return max(pool, key=lambda candidate: candidate[1])[2] if pool else None


def unused_title(slug):
    try:
        spreadsheet = brute_force_runner.get_gspread_client().open(report_name)
    except brute_force_runner.gspread.exceptions.SpreadsheetNotFound:
        return slug
    titles = {worksheet.title for worksheet in spreadsheet.worksheets()}
    return slug if slug not in titles else f"{slug}_{run_tag}"


def load_target_worksheet(sheet_name):
    try:
        spreadsheet = brute_force_runner.get_gspread_client().open(sheet_name)
        worksheet = spreadsheet.worksheet(target_title)
    except (
        brute_force_runner.gspread.exceptions.SpreadsheetNotFound,
        brute_force_runner.gspread.exceptions.WorksheetNotFound,
    ):
        return [], [], target_title, False
    values = worksheet.get_all_values()
    if not values:
        return [], [], target_title, False
    return values[0], values[1:], target_title, has_end(values)


def load_target_results(sheet_name):
    existing = {}
    header, rows, worksheet_title, end_marker = load_target_worksheet(sheet_name)
    if not header:
        return existing, worksheet_title, end_marker
    indices = brute_force_runner._get_header_indices(header)
    required = ("Segment", "Cam Master", "Cam Slave", "All MPJPE")
    if any(indices[column] == -1 for column in required):
        raise ValueError(f"Worksheet '{worksheet_title}' thiếu cột bắt buộc để resume: {required}")
    for row in rows:
        try:
            key, result = brute_force_runner._parse_history_row(row, indices)
        except (IndexError, TypeError, ValueError):
            continue
        if key and key[0] != "N/A":
            existing[key] = result
    return existing, worksheet_title, end_marker


probe_result = {
    "master": "cam0",
    "supplement": "cam1",
    "fusion_method": "proposed",
    "mpjpe": 1.0,
    "config_signature": brute_force_runner._active_config_signature(base_pipeline_config, "proposed"),
}
assert report_matches(brute_force_runner._build_report_rows({"probe": [probe_result]}), base_pipeline_config)


summary_rows = [["Configuration", "Occ. MPJPE", "Vis. MPJPE", "PA-MPJPE", "MBLE", "Accel Error (mm/frame^2)"]]
original_name_input = brute_force_runner.get_spreadsheet_name_input
original_sheet_loader = brute_force_runner._get_sheet_data
original_result_loader = brute_force_runner.load_existing_spreadsheet_results
brute_force_runner.get_spreadsheet_name_input = lambda default_name, timeout=10: report_name

try:
    for index, ablation in enumerate(ablations, start=1):
        print(f"\n===== Table 2 [{index}/{len(ablations)}]: {ablation['label']} =====")
        active_config = copy.deepcopy(base_pipeline_config)
        apply_overrides(active_config, ablation["overrides"])
        assert active_config["fusion"]["belief"]["alpha"] == float(ALPHA)
        assert active_config["fusion"]["belief"]["beta"] == float(BETA)
        write_pipeline(active_config)
        os.environ["NOTEBOOK_NAME"] = NOTEBOOK_NAME
        worksheet = find_report_worksheet(ablation["slug"], active_config)
        target_title = worksheet.title if worksheet else unused_title(ablation["slug"])
        brute_force_runner._get_sheet_data = load_target_worksheet
        brute_force_runner.load_existing_spreadsheet_results = load_target_results
        values = worksheet.get_all_values() if worksheet else []
        if has_end(values):
            print(f"[Bỏ qua] {ablation['label']} đã hoàn tất trong worksheet '{target_title}'.")
        else:
            existing_results, _, _ = load_target_results(report_name)
            completed_count = sum(result.get("mpjpe", float("inf")) != float("inf") for result in existing_results.values())
            action = f"Tiếp tục từ {completed_count} cặp" if completed_count else "Chạy mới"
            print(f"[{action}] Ghi kết quả vào worksheet '{target_title}'.")
            brute_force_runner.run_brute_force()
            spreadsheet = brute_force_runner.get_gspread_client().open(report_name)
            worksheet = spreadsheet.worksheet(target_title)
            values = worksheet.get_all_values()
        summary_rows.append([
            ablation["label"],
            mean_column(values, "Occ. MPJPE"),
            mean_column(values, "Vis. MPJPE"),
            mean_column(values, "PA-MPJPE"),
            mean_column(values, "MBLE"),
            mean_column(values, "Accel Error (mm/frame^2)"),
        ])
finally:
    write_pipeline(base_pipeline_config)
    os.environ["NOTEBOOK_NAME"] = NOTEBOOK_NAME
    brute_force_runner.get_spreadsheet_name_input = original_name_input
    brute_force_runner._get_sheet_data = original_sheet_loader
    brute_force_runner.load_existing_spreadsheet_results = original_result_loader

spreadsheet = brute_force_runner.get_gspread_client().open(report_name)
try:
    summary_sheet = spreadsheet.worksheet("Table2_Summary")
    summary_sheet.clear()
except brute_force_runner.gspread.exceptions.WorksheetNotFound:
    summary_sheet = spreadsheet.add_worksheet(title="Table2_Summary", rows="10", cols="6")
try:
    summary_sheet.update(values=summary_rows, range_name="A1")
except TypeError:
    summary_sheet.update(summary_rows)
brute_force_runner.decorate(summary_sheet, len(summary_rows), len(summary_rows[0]))
print(f"✅ Finished Table 2: {spreadsheet.url}")
